# 1.深度学习模型数据集准备
本代码实现从原始加速度CSV文件到模型可用NPZ格式数据的完整转换。主要包括：
1）读取原始数据、重采样、去噪、裁剪等预处理，保存为清洗后的CSV；
2）用滑动窗口（窗口长度50，步长25）从预处理数据中提取时间窗口；
3）分别保存有标签窗口（用于监督学习）和无标签窗口（用于预训练）。该步骤对应论文第2.1节数据集构建，为后续模型训练提供标准输入格式。


In [1]:
# =============================================================================
# 导入标准库
# os：文件和目录操作；glob：文件路径匹配（用于批量读取 CSV）
# pandas：数据处理（读取 CSV、数据清洗、合并等）
# IPython.display：在 Jupyter Notebook 中显示 DataFrame 的漂亮格式
# =============================================================================
import os


import glob
import pandas as pd
from IPython.display import display

# =============================================================================
# 将上一级目录（即项目根目录）加入 sys.path，sys.path 是 Python 的“模块搜索路径列表”
# 这样可以导入项目根目录下的 src 包中的自定义模块
# sys.dont_write_bytecode = True：禁用 .pyc 字节码文件生成，避免污染目录
# =============================================================================
import sys
sys.path.append("../")
sys.dont_write_bytecode = True

# =============================================================================
# 启用 IPython 的自动重载扩展（autoreload）
# 当修改 src 目录下的源码后，无需重启 Jupyter 内核即可自动加载最新代码
# autoreload 2 表示跟踪所有已导入模块的变化并自动重载
# 这在开发调试阶段非常有用，提高迭代效率
# =============================================================================
%load_ext autoreload
%autoreload 2

# =============================================================================
# 从 src.data_preprocess_logbot 模块导入数据预处理所需的所有函数
# 这些函数实现了论文第 2.1 节所述的完整数据预处理流水线
# =============================================================================
from src.data_preprocess_logbot import (
    get_raw_date_information,             #提取原始数据中的日期信息
    read_raw_data_and_refine_timestamp,   #读取原始 CSV 并精炼时间戳
    divide_df_if_timestamp_gap_detected,  #检测时间戳断裂并分割数据段，对应论文：加速度数据可能存在时间戳不连续的情况，需分段处理
    run_resampling_and_concat_df,         #重采样至统一采样率并拼接数据段，对应论文第 2.1 节：将 31Hz 数据上采样至 1000Hz 再下采样至 25Hz
    preprocess_sensor_data,               #传感器数据预处理（如限幅 clipping），对应论文：clipping_threshold=8，去除异常高幅值噪声
    save_preprocessed_data,               #保存预处理后的 CSV 文件
    extract_sliding_windows,              #滑动窗口提取有标签和无标签窗口，对应论文第 2.1 节：window_size=50（2秒），stride=25（1秒）
    extract_sliding_windows_v2,           #滑动窗口提取的另一种实现（用于无标签数据）
    save_labelled_windows_as_npz,         #将有标签窗口保存为 .npz 格式
    get_shuffled_list,                    #生成混洗后的索引列表
    save_blocks_of_windows_as_npz,        #将无标签窗口分块保存为 .npz 格式，对应论文第 2.5 节：无标签数据用于 CNN-AE 的无监督预训练
)

# =============================================================================
# 导入工具模块 utils
# 返回三个对象：
# - plot_parameters：绘图参数（字体大小、颜色等），可传递给 plt.rcParams.update()
# - okabe_ito_color_list：色盲友好的配色方案
# - tol_bright_color_list：Tol 明亮配色方案
# 这些配色对应论文中所有图表（图 3、图 6、图 7 等）的出版风格
# show_color_palette=False 表示不显示调色板预览图
# =============================================================================
from src import utils
(
    plot_parameters, okabe_ito_color_list, tol_bright_color_list
) = utils.setup_plot(show_color_palette=False)      # utils.setup_plot() 用于设置 matplotlib 全局绘图样式

C:\Users\bxy11\Desktop\dl-wabc\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 将路径修改为你本地的数据集根目录
data_dir = "C:/Users/bxy11/Desktop/dl-wabc/data/datasets/"
# 示例：data_dir = "./dl-wabc/raw_data"

# =============================================================================
# 数据集目录拼接
# 原始设计是将数据放在 datasets/logbot_data 子目录下
# 这里直接使用 data_dir 作为根目录，因为数据直接放在 datasets/ 下
# 这取决于实际数据存放结构，可以根据实际情况调整
# =============================================================================
# dataset_dir = f"{data_dir}/datasets/logbot_data"   # 原始设计（注释掉）
dataset_dir = f"{data_dir}"


In [3]:
# 调试模式开关：True 表示不写出文件，仅跑通流程；False 表示正常保存
# debug_test_mode = True  # 调试模式：不保存数据
debug_test_mode = 
False # 正常模式：保存数据


In [4]:
# 选择目标物种：omizunagidori（大水薙鸟） / umineko（黑尾鸥）,本代码仅对omizunagidori（大水薙鸟）进行研究
species = "omizunagidori"
# species = "umineko"


In [5]:
# 设置预处理结果的输出目录并打印
output_dir_path = f"{dataset_dir}/preprocessed_data/"
print("output directory:", output_dir_path)
print(" ")

# 获取该物种下所有原始 CSV 文件路径
path_target = f"{dat
aset_dir}/raw_data/{species}/**.csv"     # * ： 通配符，匹配任意字符， **.csv ： 匹配该文件夹下所有以 .csv 结尾的文件
raw_data_path_list = sorted(glob.glob(path_target))          # glob.glob() - 返回匹配的文件列表，第一个glob表示模块名，第二个glob表示函数名
print(f"N of raw raw_data csv files: {len(raw_data_path_list)}", )   # f-string 格式化字符串，把数字插入到文本中
for i, raw_data_path in enumerate(raw_data_path_list):       # enumerate(raw_data_path_list)：同时获取索引（i）和值（raw_data_path）
    print(f"{i:0=2}: {os.path.basename(raw_data_path)}")     # os.path.basename()：从完整路径中提取文件名（去掉前面的文件夹路径）

# =============================================================================
# 读取动物 ID 信息表（元数据表）
# 该表格包含每个文件对应的：
# - species：物种（om/um）
# - year：采集年份（2018-2022）
# - animal_tag：设备标签
# - animal_id：个体唯一编号（如 OM1803）
# - acc_sampling_rate：加速度采样率（25Hz 或 31Hz）
# - correct_timestamp：是否需要修正时间戳（布尔值）
# - back_mount：设备佩戴位置（back/abdomen）

# 对应论文第 2.1 节：设备佩戴位置差异（背部 vs 腹部）会影响加速度信号特征
# 特别是黑尾鸥中有 18 只腹部佩戴、其余背部佩戴
# 这为后续数据增强中的"旋转"操作提供了动机（论文实验 1）
# =============================================================================
animal_id_path = f"{dataset_dir}/id_files/animal_id.csv"
print("animal_id_path:", animal_id_path)
df_animal_id = pd.read_csv(animal_id_path)
display(df_animal_id.head(5))      #只显示前5行数据

output directory: C:/Users/bxy11/Desktop/dl-wabc/data/datasets//preprocessed_data/
 
N of raw raw_data csv files: 0
animal_id_path: C:/Users/bxy11/Desktop/dl-wabc/data/datasets//id_files/animal_id.csv


,animal_id,species,year,animal_tag,acc_sampling_rate,correct_timestamp,back,acc_sensor,csv_file_name
0,OM1802,omizunagidori,2018,9B24590,25,0,1,MPU-9250,Omizunagidori2018_raw_data_9B24590_lb0002.csv
1,OM1803,omizunagidori,2018,9B34075,25,0,1,MPU-9250,Omizunagidori2018_raw_data_9B34075_lb0003.csv
2,OM1804,omizunagidori,2018,9B36347,25,0,1,MPU-9250,Omizunagidori2018_raw_data_9B36347_lb0004.csv
3,OM1805,omizunagidori,2018,9B36360,25,0,1,MPU-9250,Omizunagidori2018_raw_data_9B36360_lb0005.csv
4,OM1806,omizunagidori,2018,9B36365,25,0,1,MPU-9250,Omizunagidori2018_raw_data_9B36365_lb0006.csv


In [6]:
# 遍历每个原始文件，完成：元信息读取 → 时间戳修正 → 间隔切分 → 重采样 → 裁剪 → 保存
for raw_data_path in raw_data_path_list:
    # -------------------- 步骤 1：查询元信息 --------------------
    # 从 animal_id.csv 中获取该文件对应的元数据
    # 包括：物种、年份、设备标签、个体 ID、采样率、时间戳修正标志、佩戴位置
    # 对应论文第 2.1 节：
    #   - 部分数据采样率为 31Hz，需要重采样至 25Hz
    #   - 黑尾鸥中
    有 18 只腹部佩戴、其余背部佩戴
    #   - 佩戴位置差异会影响加速度信号，是后续数据增强中"旋转"操作的动机
    (
        species,
        year,
        animal_tag,
        animal_id,
        acc_sampling_rate,
        correct_timestamp,
        back_mount
    ) = get_raw_date_information(raw_data_path,
                                 animal_id_path)

    # -------------------- 步骤 2：读取原始数据并修正时间戳 --------------------
    # 读取原始 CSV 文件，返回包含以下列的 DataFrame：
    #   - datetime：日期时间（字符串格式）
    #   - unixtime：Unix 时间戳（浮点数，单位：秒）
    #   - acc_x, acc_y, acc_z：三轴加速度（单位：g）
    #   - label：行为标签文本（如 "stationary"）
    #   - label_id：行为标签数字编号
    # correct_timestamp 为 True 时，对时间戳进行修正（如校准初始时间）
    df = read_raw_data_and_refine_timestamp(raw_data_path,
                                            correct_timestamp)

    # -------------------- 步骤 3：检测时间戳断裂并切分数据段 --------------------
    # 检查时间序列中是否存在长时间的数据缺口
    # 如果相邻时间戳间隔超过阈值（gap_min_limit=5 秒），则在该处切分数据
    # 为什么需要切分？
    #   - 设备可能因信号丢失、重启等原因产生数据缺失
    #   - 如果在缺失区域直接进行重采样插值，会引入虚假数据
    #   - 切分后每段数据内部时间连续，可独立重采样
    #
    # gap_min_limit=5 表示 5 秒阈值
    # 内部计算：acc_sampling_rate × gap_min_limit = 25 × 5 = 125 个采样点
    # 即相邻时间戳差距超过 125 个采样间隔时触发切分
    df_list = divide_df_if_timestamp_gap_detected(df,
                                                  acc_sampling_rate,
                                                  gap_min_limit=5)

    # -------------------- 步骤 4：重采样并拼接数据段 --------------------
    # 对每段数据分别进行重采样，然后拼接为完整的时间序列
    # 重采样逻辑（对应论文第 2.1 节）：
    #   1. 如果原始采样率为 31Hz：
    #      - 先用线性插值上采样到 1000Hz（高分辨率，便于精确重采样）
    #      - 再下采样到目标采样率 25Hz（因为 31 不是 25 的倍数，无法直接重采样）
    #   2. 如果原始采样率已是 25Hz：
    #      - 直接按 25Hz 重采样（确保时间轴规整）
    #
    # remove_sec=3：去除每段开头和结尾各 3 秒数据
    #   目的：消除重采样时的边界效应（边界处插值可能不准确）
    #
    # check_df=False：不打印详细的检查信息（减少输出）
    df = run_resampling_and_concat_df(df_list,
                                      acc_sampling_rate,
                                      remove_sec=3,
                                      check_df=False)

    # -------------------- 步骤 5：传感器数据预处理 --------------------
    # 对加速度数据进行预处理，去除异常值
    # clipping=True：启用限幅裁剪
    # clipping_threshold=8：阈值设为 8（单位：g，即重力加速度）
    #   目的：去除设备碰撞、电磁干扰等产生的异常高幅值噪声
    #   原理：将绝对值超过 8g 的采样点裁剪到 ±8g 范围内
    # method="none"：不进行额外的滤波处理（如低通、高通滤波）
    #   注意：论文中用于深度学习模型的是原始加速度数据（raw acceleration data）
    #   而用于传统机器学习的手工特征（119 features）则在后续步骤中单独提取
    # 对应论文第 2.1 节：加速度数据存在设备噪声，需要预处理
    df = preprocess_sensor_data(df,
                                clipping=True,
                                clipping_threshold=8,
                                method="none",
                                check_df=False)

    # -------------------- 步骤 6：保存预处理结果 --------------------
    # 根据 debug_test_mode 决定是否实际保存文件
    # debug_test_mode == True  ：调试模式，仅打印信息，不写入磁盘
    #   用途：快速验证流程，避免覆盖已有数据或产生大量临时文件
    # debug_test_mode == False ：正常模式，调用 save_preprocessed_data 保存
    #   保存路径：preprocessed_data/[species]/[animal_id].csv
    #   文件内容：包含 datetime, unixtime, acc_x, acc_y, acc_z, label, label_id
    # 对应论文第 2.1 节：预处理后的 CSV 用于下一步滑动窗口提取
    if debug_test_mode == True:
        print(f"| debug mode -> do not save raw_data |")
    else:
        save_preprocessed_data(df,
                               output_dir_path,
                               species,
                               animal_id)

print(f"-----------------------------------")
print(f"raw raw_data preprocessing completed !")
print(f"-----------------------------------")

-----------------------------------
raw raw_data preprocessing completed !
-----------------------------------


## 2. 将预处理后的 CSV 文件转换为 NPZ 文件
提取滑动窗口并保存为 npz 格式

### 2a. 提取有标签数据

In [7]:
# 选择目标物种

species = "omizunagidori"
# species = "umineko"


In [8]:
# 列出该物种下所有已预处理的 CSV 文件
target_path = f"{dataset_dir}/preprocessed_data/{species}/**.csv"
preprocessed_data_path_list = sorted(glob.glob(target_path))    # sorted() 按文件名排序，确保输出顺序一致（便于对照）
print("input_dir: ")
counter = 0
for prepro
cessed_data_path in preprocessed_data_path_list:
    print(str(counter).zfill(2), ": ", os.path.basename(preprocessed_data_path))  # str(counter).zfill(2) 将数字补齐为两位（如 0 → "00"）
    counter = counter + 1
print("Length of raw_data_path_list", len(preprocessed_data_path_list))      # 输出该物种下已预处理的个体数量

input_dir: 
00 :  OM1802.csv
01 :  OM1803.csv
02 :  OM1804.csv
03 :  OM1805.csv
04 :  OM1806.csv
05 :  OM1807.csv
06 :  OM1808.csv
07 :  OM1809.csv
08 :  OM1810.csv
09 :  OM1811.csv
10 :  OM1901.csv
11 :  OM2001.csv
12 :  OM2002.csv
13 :  OM2003.csv
14 :  OM2005.csv
15 :  OM2006.csv
16 :  OM2101.csv
17 :  OM2102.csv
18 :  OM2103.csv
19 :  OM2201.csv
20 :  OM2202.csv
21 :  OM2203.csv
22 :  OM2204.csv
23 :  OM2205.csv
24 :  OM2206.csv
25 :  OM2207.csv
26 :  OM2208.csv
27 :  OM2209.csv
28 :  OM2210.csv
29 :  OM2211.csv
30 :  OM2212.csv
31 :  OM2213.csv
32 :  OM2214.csv
Length of raw_data_path_list 33


In [9]:
# 有标签窗口 npz 文件的输出根目录
labelled_data_base_dir = f"{dataset_dir}/npz_format/labelled/{species}/"

print("Extract sliding windows from preprocessed raw_data (.csv) and save them as .npz files")
for preprocessed_da
ta_path in preprocessed_data_path_list:
    print("-----------------------------------------------------------------------")
    animal_id = os.path.basename(preprocessed_data_path).replace(".csv", "")  # 从文件路径中提取文件名（如 "OM1803.csv"），去掉 .csv 后缀得到个体 ID
    print(animal_id, end=": ")

  # -------------------- 滑动窗口提取 --------------------
    (
        X_list,                  #所有提取的窗口数据列表（含标签和无标签）
        label_id_list,           #对应窗口的标签 ID（无标签窗口为 NaN）
        timestamp_list,          #对应窗口的时间戳
        labelled_flag_list,      #标记窗口是否有标签（True/False）
        labelled_X_list,         #仅有标签的窗口数据列表（用于监督学习）
        labelled_label_id_list,  #对应有标签窗口的标签 ID
        labelled_timestamp_list, #对应有标签窗口的时间戳
        timestamp_gap_idx_list   #时间戳断裂的窗口索引列表
    ) = extract_sliding_windows(preprocessed_data_path,      # 调用 extract_sliding_windows 函数，从预处理 CSV 中提取时间窗口
                                sliding_window_size=50,      # 窗口长度 50 个采样点（对应 2 秒，采样率 25Hz）
                                sliding_window_step_size=25) # 步长 25（对应 1 秒，即 50% 窗口重叠）
    print(f"N of extracted windows: {len(X_list)}")
    print(f"N of labelled windows:  {len(labelled_X_list)}")
    print(f"N of timestamp gaps:    {len(timestamp_gap_idx_list)}")

 # -------------------- 保存有标签窗口为 .npz 格式 --------------------
    # 只有当该个体存在有标签窗口时才进行保存
    # 有些个体可能没有视频标注数据，因此 labelled_X_list 为空
    if len(labelled_X_list) > 0:
        # 构建该个体的 npz 输出目录
        # 例如：npz_format/labelled/omizunagidori/OM1803/
        npz_file_dir = labelled_data_base_dir + animal_id + "/"
        print("Saving labelled windows as npz ...")

        # debug_test_mode == True  ：仅打印信息，不写入磁盘
        # debug_test_mode == False ：调用 save_labelled_windows_as_npz 保存
        # 对应论文第 2.1 节：NPZ 格式是深度学习模型的标准输入格式
        # 也是后续训练代码（run_dl_model_training）直接读取的数据格式
        if debug_test_mode == True:
            print(f"| debug mode -> do not save raw_data |")
        else:
            save_labelled_windows_as_npz(animal_id,
                                         npz_file_dir,
                                         labelled_X_list,
                                         labelled_label_id_list,
                                         labelled_timestamp_list)

print(f"----------------------------------------")
print(f"Labelled window extraction completed !")
print(f"----------------------------------------")

Extract sliding windows from preprocessed raw_data (.csv) and save them as .npz files
-----------------------------------------------------------------------
OM1802: length of df:  1167175
Extracting sliding windows ...
N of extracted windows: 46647
N of labelled windows:  0
N of timestamp gaps:    0
-----------------------------------------------------------------------
OM1803: length of df:  1008000
Extracting sliding windows ...
N of extracted windows: 40290
N of labelled windows:  1704
N of timestamp gaps:    0
Saving labelled windows as npz ...
-----------------------------------------------------------------------
OM1804: length of df:  588450
Extracting sliding windows ...
N of extracted windows: 23520
N of labelled windows:  1425
N of timestamp gaps:    0
Saving labelled windows as npz ...
-----------------------------------------------------------------------
OM1805: length of df:  1195575
Extracting sliding windows ...
N of extracted windows: 47744
N of labelled windows:  290

### 2b. 提取无标签数据 v2

In [10]:
# 选择目标物种
species = "omizunagidori"
# species = "umineko"


In [11]:
# 列出该物种下所有已预处理的 CSV 文件
target_path = f"{dataset_dir}/preprocessed_data/{species}/**.csv"
preprocessed_data_path_list = sorted(glob.glob(target_path))
print("input_dir: ")
counter = 0
for preprocessed_data_path in preprocessed_data_path_list:
    print(str(counter).zfill(2), ": ", os.path.basename(preprocessed_data_path))
    counter = counter + 1
print("Length of raw_data_path_list", len(preprocessed_data_path_list))

input_dir: 
00 :  OM1802.csv
01 :  OM1803.csv
02 :  OM1804.csv
03 :  OM1805.csv
04 :  OM1806.csv
05 :  OM1807.csv
06 :  OM1808.csv
07 :  OM1809.csv
08 :  OM1810.csv
09 :  OM1811.csv
10 :  OM1901.csv
11 :  OM2001.csv
12 :  OM2002.csv
13 :  OM2003.csv
14 :  OM2005.csv
15 :  OM2006.csv
16 :  OM2101.csv
17 :  OM2102.csv
18 :  OM2103.csv
19 :  OM2201.csv
20 :  OM2202.csv
21 :  OM2203.csv
22 :  OM2204.csv
23 :  OM2205.csv
24 :  OM2206.csv
25 :  OM2207.csv
26 :  OM2208.csv
27 :  OM2209.csv
28 :  OM2210.csv
29 :  OM2211.csv
30 :  OM2212.csv
31 :  OM2213.csv
32 :  OM2214.csv
Length of raw_data_path_list 33


In [12]:
# 含全部窗口的 npz 输出根目录（每个 npz 内含 20 个打乱顺序的窗口）
unlabelled_data_base_dir = f"{dataset_dir}/npz_format/shuffled_20_v2/{species}/"
print("Extract sliding windows from preprocessed raw_data (.csv) and save them as .npz files")

for preprocessed_data_path in preprocessed_data_path_list:
    print("-----------------------------------------------------------------------")
    animal_id = os.path.basename(preprocessed_data_path).replace(".csv", "")
    print(animal_id, end=": ")

    # 使用 v2 版本提取滑动窗口：包含全部窗口（含标签与无标签）
    (
        X_list,
        label_id_list,
        timestamp_list,
        labelled_flag_list,
        labelled_X_list,
        labelled_label_id_list,
        labelled_timestamp_list,
        timestamp_gap_idx_list
    ) = extract_sliding_windows_v2(
        preprocessed_data_path,
        sliding_window_size=50,
        sliding_window_step_size=25
    )
    print(f"N of extracted windows: {len(X_list)}")
    print(f"N of labelled windows:  {len(labelled_X_list)}")
    print(f"N of timestamp gaps:    {len(timestamp_gap_idx_list)}")


    # 随机打乱所有窗口顺序，便于后续按块存储到 npz
    (
        index_list_random,          # 打乱后的索引顺序
        X_list_random,              # 打乱后的窗口数据
        label_id_list_random,       # 打乱后的标签ID
        timestamp_list_random,      # 打乱后的时间戳
        labelled_flag_list_random   # 打乱后的标签标志
    ) = get_shuffled_list(X_list,            # get_shuffled_list 返回打乱后的索引和对应的数据列表
                          label_id_list,
                          timestamp_list,
                          labelled_flag_list,
                          random_seed=558)

    # 将所有窗口保存为 npz（每个文件 20 个窗口）
    num_windows_per_npz_file = 20
    npz_file_dir = unlabelled_data_base_dir + animal_id + "/"
    print(f"npz_file_dir: {npz_file_dir}")
    print("Saving all windows as npz ...")
    if debug_test_mode == True:
        print(f"| debug mode -> do not save raw_data |")
    else:
        save_blocks_of_windows_as_npz(num_windows_per_npz_file,
                                      animal_id,
                                      npz_file_dir,
                                      index_list_random,
                                      X_list_random,
                                      label_id_list_random,
                                      timestamp_list_random,
                                      labelled_flag_list_random)

print(f"----------------------------------------")
print(f"Unlabelled window extraction completed !")
print(f"----------------------------------------")

Extract sliding windows from preprocessed raw_data (.csv) and save them as .npz files
-----------------------------------------------------------------------
OM1802: length of df:  1167175
Extracting sliding windows ...
N of extracted windows: 46647
N of labelled windows:  0
N of timestamp gaps:    0
npz_file_dir: C:/Users/bxy11/Desktop/dl-wabc/data/datasets//npz_format/shuffled_20_v2/omizunagidori/OM1802/
Saving all windows as npz ...
-----------------------------------------------------------------------
OM1803: length of df:  1008000
Extracting sliding windows ...
N of extracted windows: 40290
N of labelled windows:  1704
N of timestamp gaps:    0
npz_file_dir: C:/Users/bxy11/Desktop/dl-wabc/data/datasets//npz_format/shuffled_20_v2/omizunagidori/OM1803/
Saving all windows as npz ...
-----------------------------------------------------------------------
OM1804: length of df:  588450
Extracting sliding windows ...
N of extracted windows: 23520
N of labelled windows:  1425
N of timest